In [ ]:
%load_ext autoreload
%autoreload 2
from reward_relative.path_dict_seahorse import path_dictionary as path_dict
from reward_relative import utilities as ut
from reward_relative import plotUtils as pt
from reward_relative import spatial
from reward_relative import placeCellPlot
from reward_relative import dayData as dd
from reward_relative import behavior
import pickle
import dill
import numpy as np
import os 
import matplotlib.pyplot as plt
import TwoPUtils
import protter_functions as pf
from scipy.signal import medfilt
from matplotlib.lines import Line2D
import pandas as pd
import umap
import hdbscan
import importlib
import scipy.interpolate as interp

In [ ]:
animal = 'GCAMP14'
day = 5

hd5_animal = str(animal)
hd5_day = str(day) 

hd5_df = pf.recreate_trial_meta_df(f[hd5_animal][hd5_day]['trial_metadata'])




cols_to_map = ['idx', 'hdb_labels', 'swap_zone_start', 'swap_zone_end']
idxs = metadata.loc[(metadata.animal == animal)&(metadata.day == day), cols_to_map+['trial']]

for col in ['idx', 'hdb_labels', 'swap_zone_start', 'swap_zone_end']:
    hd5_df[col] = hd5_df['trial'].map(idxs.set_index('trial')[col])



fig, ax = plt.subplots()
plot_raster_from_licks(f[hd5_animal][hd5_day]['lick_data']['smoothed_licks'][[i for i in range(len(hd5_df))]], hd5_df, ax)
# plot_raster_from_licks(f[animal][day]['lick_data']['smoothed_licks'][[9,10,29,30,34,35,55]], df.iloc[[9,10,29,30,34,35,55]], ax)

In [ ]:
%matplotlib widget
import protter_plot_functions as ppf
ppf = importlib.reload(ppf)
from protter_plot_functions import add_hover, add_color_selector, ColorSelector, ColorSelectorV2#, #add_selectors
from claude_test_space import add_selectors
from protter_plot_functions import TrialPlotLinker

bin_centers = np.arange(5,455,10)

# display_slicer = (trials_na_free.animal == 'GCAMP12')&(trials_na_free.day == 14)

animal = 'GCAMP14'
day = 12

display_slicer = (trials_na_free.animal == animal) &(trials_na_free.day == day)
#
# display_slicer = (trials_na_free.trial_type == 'post_swap')
# display_slicer = [True]*len(licks_na_free)

# feats = get_lick_features(licks_na_free[display_slicer], bin_centers, trials_na_free[display_slicer], norm = False)


features, sorted_meta, sorted_licks = generate_cross_day_features(licks_na_free[display_slicer], 
                                                                  trials_na_free[display_slicer], 
                                                                  bin_centers, norm = False, 
                                                                  reward_relative=False
                                                                  )
feats = features[[c for c in features.columns if not c in [ 'n_licks', ]]]
feats = np.nan_to_num(feats)


reducer = umap.UMAP(n_neighbors=int(sum(display_slicer/22)), min_dist=0)

transform = reducer.fit_transform(feats)


# lick_pca = skd.PCA()
# transform = lick_pca.fit_transform(feats[:,1:])

min_samples = 3
min_cluster_size = 3
new_labels = hdbscan.HDBSCAN(
    min_samples=min_samples,
    min_cluster_size=min_cluster_size,

).fit_predict(transform)


metadata = sorted_meta.copy()
metadata['hdb_labels'] = new_labels


scatter_meta = metadata
scatter_data = transform
linker = TrialPlotLinker(metadata)
raster_data = sorted_licks

fig = plt.figure(figsize = (11,5))
fig.suptitle('pre_swap vs other trial')
axes_dict = fig.subplot_mosaic([['lick_umap', 'lick_umap', 'lick_raster'],
                                    ['lick_umap','lick_umap', 'lick_raster'],
                                    ],
                                height_ratios = [3.0,2.0],
                                width_ratios = [2.0,2.0,3])

ax_umap= axes_dict['lick_umap']




scatter_1 = ax_umap.scatter(scatter_data[:,0], scatter_data[:,1], alpha = 0.5, label = 'preswap')

ax_umap.legend()


add_hover(scatter_1, scatter_meta)
color_selector = ColorSelectorV2(scatter_1,scatter_meta)

sel = add_selectors(scatter_1, linker, scatter_meta)


umap_view = ppf.ScatterView(data = transform, meta = scatter_meta, plt_obj = scatter_1)


lick_ax = axes_dict['lick_raster']
lick_ax.tick_params(right=True, labelright=True, left=False, labelleft=False)

raster_view = ppf.LickRasterView(raster_data, bin_centers, scatter_meta, lick_ax)
raster_view.link_colormaps(color_selector.get_color_linker(), )
raster_view.sort_by({'hdb_labels':'descending', 'reward_zone_start':'descending','day':'descending', 'trial':'ascending'})
color_selector.add_after_recolor_call(raster_view.update_linked_colors)

linker.add_views([umap_view, raster_view]) 
# raster_view.update(start_idx)
# umap_view.update(start_idx)
